In [1]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")

In [16]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

In [17]:
def run_query(query, parameters=None):
    records, summary, keys = driver.execute_query(query, parameters_=parameters or {})
    return [record.data() for record in records]

In [18]:
query = """create(s:student{name:$name, age:$age, city:$city}) RETURN s"""
result = run_query(query, parameters={"name": "Vineet Patel", "age": 24, "city": "Mumbai"})

In [19]:
result

[{'s': {'city': 'Mumbai', 'name': 'Vineet Patel', 'age': 24}}]

In [20]:
run_query(query, parameters={"name": "Pavan", "age": 29, "city": "Bangalore"})
run_query(query, parameters={"name": "Hitesh", "age": 29, "city": "Bangalore"})
run_query(query, parameters={"name": "Riya", "age": 23, "city": "Palghar"})
run_query(query, parameters={"name": "Sanika", "age": 24, "city": "Mumbai"})

[{'s': {'city': 'Mumbai', 'name': 'Sanika', 'age': 24}}]

In [21]:
students = [
    {
        "student_id": "S001",
        "name": "Rahul",
        "age": 24,
        "city": "Bengaluru"
    },
    {
        "student_id": "S002",
        "name": "Priya",
        "age": 23,
        "city": "Delhi"
    },
    {
        "student_id": "S003",
        "name": "Aman",
        "age": 27,
        "city": "Mumbai"
    },
    {
        "student_id": "S004",
        "name": "Sneha",
        "age": 25,
        "city": "Bengaluru"
    }
]

In [22]:
query = """CREATE (s:student {student_id: $student_id, name: $name, age: $age, city: $city}) RETURN s"""

for student in students:
    result = run_query(query, parameters=student)
    print(result)

[{'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24}}]
[{'s': {'city': 'Delhi', 'name': 'Priya', 'student_id': 'S002', 'age': 23}}]
[{'s': {'city': 'Mumbai', 'name': 'Aman', 'student_id': 'S003', 'age': 27}}]
[{'s': {'city': 'Bengaluru', 'name': 'Sneha', 'student_id': 'S004', 'age': 25}}]


In [23]:
courses = [
    {
        "course_id": "C001",
        "name": "Agentic AI"
    },
    {
        "course_id": "C002",
        "name": "Data Science"
    },
    {
        "course_id": "C003",
        "name": "Backend Engineering"
    }
]

In [24]:
query = "CREATE (c:course {course_id: $course_id, name: $name}) RETURN c"

for course in courses:
    result = run_query(query, parameters=course)
    print(result)

[{'c': {'course_id': 'C001', 'name': 'Agentic AI'}}]
[{'c': {'course_id': 'C002', 'name': 'Data Science'}}]
[{'c': {'course_id': 'C003', 'name': 'Backend Engineering'}}]


In [25]:
skills = [
    "Python",
    "Machine Learning",
    "RAG",
    "Neo4j",
    "FastAPI",
    "LangGraph"
]

In [28]:
query= """create(s:skill{name:$name}) RETURN s"""
for skill in skills:
    result = run_query(query, parameters={"name": skill})
    print(result)

[{'s': {'name': 'Python'}}]
[{'s': {'name': 'Machine Learning'}}]
[{'s': {'name': 'RAG'}}]
[{'s': {'name': 'Neo4j'}}]
[{'s': {'name': 'FastAPI'}}]
[{'s': {'name': 'LangGraph'}}]


In [29]:
mentor = {
    "mentor_id": "M001",
    "name": "Sudhanshu",
    "company": "Euron"
}

In [30]:
query = """create(m:mentor{mentor_id:$mentor_id,name:$name,company:$company}) RETURN m"""
run_query(query, parameters=mentor)

[{'m': {'mentor_id': 'M001', 'name': 'Sudhanshu', 'company': 'Euron'}}]

In [31]:
query = """MATCH (s:student {student_id: $student_id})  
MATCH (c:course {course_id: $course_id})
MERGE (s)-[:ENROLLED_IN]->(c)
RETURN s, c
"""

run_query(query, parameters={"student_id": "S001", "course_id": "C001"})

[{'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24},
  'c': {'course_id': 'C001', 'name': 'Agentic AI'}}]

In [32]:
query = """MATCH (s:student {student_id: $student_id})  
MATCH (c:course {course_id: $course_id})
MERGE (s)-[r:ENROLLED_IN]->(c)
SET r.enrolled_on = date($enrolled_on),
    r.status = $status,
    r.grade = $grade,
    r.feedback = $feedback
RETURN s, c
"""
run_query(query, parameters={
    "student_id": "S001", 
    "course_id": "C001", 
    "enrolled_on": "2026-09-19", 
    "status": "active", 
    "grade": "A", 
    "feedback": "Excellent performance!"})

[{'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24},
  'c': {'course_id': 'C001', 'name': 'Agentic AI'}}]

In [33]:
enrollments = [
    {"student" : "S001", "course" : "C001"},
    {"student" : "S002", "course" : "C002"},
    {"student" : "S003", "course" : "C003"},
    {"student" : "S004", "course" : "C003"},
    {"student" : "S005", "course" : "C002"},
]

In [ ]:
query = """MATCH (s:student {student_id:$student_id})
MATCH (c:course {course_id:$course_id})
MERGE (s)-[r:ENROLLED_IN]->(c)
RETURN s, c
"""

for enrollment in enrollments:
    run_query(query, parameters={
        "student_id": enrollment["student"],
        "course_id": enrollment["course"],
    })

In [35]:
student_skills = [
    {"student": "S001", "skill": "Python"},
    {"student": "S001", "skill": "RAG"},
    {"student": "S001", "skill": "LangGraph"},

    {"student": "S002", "skill": "Python"},
    {"student": "S002", "skill": "Neo4j"},

    {"student": "S003", "skill": "Python"},
    {"student": "S003", "skill": "Machine Learning"},

    {"student": "S004", "skill": "FastAPI"},
    {"student": "S004", "skill": "Python"}
]

query = """MATCH (s:student {student_id:$student_id})
MATCH (sk:skill {name:$skill})
MERGE (s)-[r:HAS_SKILL]->(sk)
RETURN s, sk
"""

for student_skill in student_skills:
    run_query(query, parameters={
        "student_id": student_skill["student"],
        "skill": student_skill["skill"]
    })


In [36]:
query = """MATCH (s:student {name:$name}) -[HAS_SKILL]->(sk:skill) RETURN s, sk"""
run_query(query, parameters={"name": "Rahul"})

[#C25A]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-27-0122.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))): OSError('No data')
Transaction failed and will be retried in 0.9100370969109569s (Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-27-0122.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv4Address(('34.124.169.171', 7687))))


[{'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24},
  'sk': {'name': 'Python'}},
 {'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24},
  'sk': {'name': 'RAG'}},
 {'s': {'city': 'Bengaluru', 'name': 'Rahul', 'student_id': 'S001', 'age': 24},
  'sk': {'name': 'LangGraph'}}]

In [38]:
query = """MATCH (s:student where s.age <= $age and s.city = $city) RETURN s.name, s.age"""
run_query(query, parameters={"age": 25, "city": "Mumbai"})

[{'s.name': 'Vineet Patel', 's.age': 24}, {'s.name': 'Sanika', 's.age': 24}]

In [39]:
query = """MATCH (s:student {name:$name}) DETACH DELETE s"""
run_query(query, parameters={"name": "Rahul"})

[]